In [2]:
from ngsolve import *
from ngsolve.webgui import Draw
import numpy as np
import scipy.optimize
import csv


class gel_debonded2D:
    def __init__(self, length, thickness, shear_modulus, entropic_unit, phi0, chi, mass_density):
        self.L = length      # measured in mm
        self.d = thickness    # measured in mm
        self.G = shear_modulus               # measured in MPa
        self.entropic_unit = entropic_unit  # measured in MPa
        self.phi0 = phi0
        self.chi =  chi  
        self.mass_density = mass_density

        self.gamma = self.G/self.entropic_unit # 0.0009516837481698391 self.compute_gamma( lamb =1.4874):
        self.lambda_target = 1.99  # 0.13/136.6 = gammafun(1.99)  up to rounding errors               

        gel = self
        
        
        self.lambda_iso = (scipy.optimize.fsolve(self.auxIsotropic, 1.7))[0]        
        lambda_iso = gel.lambda_iso
        self.reference_energy_density = self.auxEnergyDensity(lambda_iso, lambda_iso, lambda_iso)                
        
        self.lambdaUniaxial = (scipy.optimize.fsolve(self.auxFunctionUniaxial, 2))[0]
        lambdaUniaxial = gel.lambdaUniaxial
        self.wUniaxial = self.auxEnergyDensity(1, lambdaUniaxial, 1) - gel.reference_energy_density
        
        self.lambdaEquiBiaxial = (scipy.optimize.fsolve(self.auxFunctionEquibiaxial, 2))[0]
        lambdaEquiBiaxial = gel.lambdaEquiBiaxial
        self.wEquiBiaxial = self.auxEnergyDensity(lambdaEquiBiaxial, lambdaEquiBiaxial, 1) - gel.reference_energy_density

    def auxIsotropic(self,s):
        gel = self
        return s*gel.dH(s*s*s) + gel.gamma
    def auxFunctionUniaxial(self,s):
        gel = self
        return gel.dH(s) + gel.gamma*s
    def auxFunctionEquibiaxial(self,s):
        gel = self
        return gel.dH(s*s) + gel.gamma
    def auxEnergyDensity(self, lambda1, lambda2, lambda3):
        gel=self
        phi0=gel.phi0;
        G=gel.G;
        chi=gel.chi;
        nu=gel.entropic_unit

        J= lambda1*lambda2*lambda3;
        phi = phi0/J;
        return 0.5*G*(lambda1**2 + lambda2**2 + lambda3**2) + nu*((J-phi0)*np.log(1-phi) + phi0*chi*(1-phi))
        
    def phi(self, J):
        return self.phi0/J

    def H(self, J):
        return (J - self.phi0)*log(1-self.phi(J))  + self.phi0 * self.chi*(1-self.phi(J))

    def dH(self, J):
        return self.phi(J) + np.log(1-self.phi(J)) + self.chi * self.phi(J)**2
    
    # compute gamma using uniaxial approximation
    # def gammafun(self,lamb):
    #    return -self.dH(lamb)/lamb

    def Gfun(self, lamb):
        nu = self.entropic_unit
        return (-self.dH(lamb)/lamb)*nu
        
    # energy density
    def W(self, F):
        J = Det(F)
        C = F.trans* F
        G = self.G
        nu = self.entropic_unit
        reference_energy_density =  self.reference_energy_density
        return 0.5*G *( Trace(C) +1) + nu*(self.H(J)) - reference_energy_density

    
### Main ###

# More stable parameters
G = 0.13               # measured in MPa
entropic_unit = 136.6  # k_B*T/V_m measured in MPa
phi0 = 0.2
chi =  0.348           # J. Elast. 2022, Sect. 3.4
rho = 1.23e-6          # measured in kg/mm³
g = 9.8                # measured in m/s²
AA = 1e5

# Most commonly changed parameters
L= 90.0
d = 1.62
order = 3

first_index_delta = 14
last_index_delta = 15
indexes_deltas = range(first_index_delta, last_index_delta+1)
# indexes_deltas = range(0,9+1)
print(f'indexes_deltas = {indexes_deltas}')

##
gel = gel_debonded2D(length=L, thickness=d, shear_modulus = G,\
                     entropic_unit=entropic_unit, phi0=phi0, chi=chi, mass_density=rho)
gravity = CoefficientFunction((0,-g))

lambdaUniaxial = gel.lambdaUniaxial
lambdaEquiBiaxial = gel.lambdaEquiBiaxial
wUniaxial = gel.wUniaxial
wEquiBiaxial = gel.wEquiBiaxial

##
folder_name_suffix = str(int(d)) + '_' + str(int(d%1*100)).zfill(2)
delta_values = np.loadtxt('meshes' + folder_name_suffix + '/deltas')

for index_delta in indexes_deltas:
    mesh_file = 'meshes' + folder_name_suffix + '/mesh{}.vol'.format(index_delta);
    mesh = Mesh(mesh_file)
    
    delta = delta_values[index_delta];
    print('------')
    print("Mesh number = {}, delta={:.3f}".format(index_delta,delta))

    # fes = VectorH1(mesh, order=order, dirichlet="bonded_interface", dirichlety="debonded_interface")
    # fes = VectorH1(mesh, order=order, dirichlet="bonded_interface")
    fes = VectorH1(mesh, order=order)
    print('Ndof: ', fes.ndof)

    filename_suffix = "_d={:.2f}_delta={:.3f}".format(d, delta)
    filename='gridfunctions'+folder_name_suffix+\
                '/result_debonded2D'+filename_suffix+\
                "_order={}".format(order)\

    u = GridFunction(fes)
    u.Load(filename + '.gfu')
    #Draw(u_noExtraBC, deformation=True)
    F = Id(2) + Grad(u)
    print('Average energy density [MPa] = ', round(Integrate(gel.W(F), mesh)/(L*d),3))   

    numericalEnergy = Integrate(gel.W(F) - rho*InnerProduct(gravity, u), mesh, order=10)
    print('Total energy [mJ/mm]: {:.2f}'.format(numericalEnergy))    

    gravitationalEnergy = + rho*g*L*(d**2)/2*(delta*(lambdaUniaxial-1) + (1-delta)*(lambdaEquiBiaxial-1))
    theoreticalEnergy = gravitationalEnergy + (((L*delta)*d*wUniaxial + (L*(1-delta))*d*wEquiBiaxial))
    print('Theoretical energy [mJ/mm]: {:.2f}'.format(theoreticalEnergy))

    print('(Equi-biaxial energy density = {:.3f}. Uniaxial energy density = {:.3f}.)'.format(wEquiBiaxial, wUniaxial))
    L=gel.L
    d=gel.d
    deformed_length = (u(mesh(L/2,d)))[0] - (u(mesh(-L/2,d)))[0] + L
    print('Deformed length = {:.2f} [mm]'.format(deformed_length) )
    
    aux_thickness = np.array(u(mesh(gel.L/2, gel.d)))
    deformed_thickness = aux_thickness[1]+d
    print('Deformed thickness at the right = {:.2f} [mm]'.format(deformed_thickness))
    
    # Draw(mesh)
    Draw(gel.W(F), mesh, deformation=u, min = 0.03, max=0.17)

    print(' ')


indexes_deltas = range(14, 16)
------
Mesh number = 14, delta=0.163
Ndof:  16670
Average energy density [MPa] =  0.061
Total energy [mJ/mm]: 8.90
Theoretical energy [mJ/mm]: 9.51
(Equi-biaxial energy density = 0.048. Uniaxial energy density = 0.152.)
Deformed length = 141.31 [mm]
Deformed thickness at the right = 2.68 [mm]


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

 
------
Mesh number = 15, delta=0.172
Ndof:  16670
Average energy density [MPa] =  0.062
Total energy [mJ/mm]: 9.04
Theoretical energy [mJ/mm]: 9.66
(Equi-biaxial energy density = 0.048. Uniaxial energy density = 0.152.)
Deformed length = 140.76 [mm]
Deformed thickness at the right = 2.68 [mm]


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

In [5]:
print(wUniaxial, wEquiBiaxial, rho*g*d)
print(rho*g*d/wUniaxial)
sigma = 1100e-3
d_max = sigma/(wUniaxial -wEquiBiaxial)
print(d_max )

0.15205895574391803 0.0483467346136095 3.6162000000000004e-05
0.00023781565395530077
10.606271739353767
